In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import zipfile, os
zip_path = "/content/drive/MyDrive/ButterflyClassify/butterfly_images.zip"
extract_path = "/content/ButterflyClassify/images"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Images extracted to:", extract_path)

✅ Images extracted to: /content/ButterflyClassify/images


In [10]:
import pandas as pd

image_paths, labels_list = [], []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_paths.append(os.path.join(root, file))
            labels_list.append(os.path.basename(root))  # folder name = species

df = pd.DataFrame({"image": image_paths, "label": labels_list})
print("✅ Dataset ready with", len(df), "images and", len(df['label'].unique()), "classes.")

✅ Dataset ready with 9285 images and 2 classes.


In [11]:
!pip install -q gradio


In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# ===== Manual train/validation split =====
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

# ===== Image generators =====
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=None,
    x_col="image",
    y_col="label",
    target_size=(64,64),
    batch_size=16,
    class_mode="categorical",
    shuffle=True
)

val_gen = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=None,
    x_col="image",
    y_col="label",
    target_size=(64,64),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

# ===== Simple CNN model =====
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(train_gen.class_indices), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# ===== Train the model =====
history = model.fit(train_gen, validation_data=val_gen, epochs=10)

# ===== Save model =====
model.save("/content/ButterflyClassify/simple_butterfly_model.keras")
print("✅ Simple CNN model trained and saved")

Found 7428 validated image filenames belonging to 2 classes.
Found 1857 validated image filenames belonging to 2 classes.
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


465/465 ━━━━━━━━━━━━━━━━━━━━ 74s 154ms/step - accuracy: 0.6983 - loss: 0.6250 - val_accuracy: 0.7001 - val_loss: 0.6181
Epoch 2/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 73s 135ms/step - accuracy: 0.6999 - loss: 0.6136 - val_accuracy: 0.7001 - val_loss: 0.6156
Epoch 3/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 65s 140ms/step - accuracy: 0.6999 - loss: 0.6085 - val_accuracy: 0.7001 - val_loss: 0.6162
Epoch 4/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 62s 132ms/step - accuracy: 0.6999 - loss: 0.5981 - val_accuracy: 0.7001 - val_loss: 0.6212
Epoch 5/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 62s 132ms/step - accuracy: 0.7038 - loss: 0.5751 - val_accuracy: 0.7006 - val_loss: 0.6372
Epoch 6/10
465/465 ━━━━━━━━━━━━━━━━━━━━ 63s 135ms/step - accuracy: 0.7255 - loss: 0.5286 - val_accuracy: 0.6613 - val_loss: 0.6953
Epoch 7/10


In [ ]:
import gradio as gr
import numpy as np

# Make sure 'labels' and 'model' are already defined from your training step

def predict_butterfly(img):
    img = img.convert("RGB").resize((64,64))
    img_array = np.array(img)/255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0]
    confidence = np.max(prediction) * 100
    species_index = np.argmax(prediction)
    species = labels[species_index]

    if confidence < 80:
        return img, "⚠️ Not a butterfly image", "Prediction confidence too low"

    return img, f"🦋 {species}", f"{confidence:.2f}% confidence"

demo = gr.Interface(
    fn=predict_butterfly,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=[
        gr.Image(label="Uploaded Image"),
        gr.Textbox(label="Prediction"),
        gr.Textbox(label="Confidence")
    ],
    title="🦋 Butterfly Classifier",
    description="Upload a butterfly image and see the species and confidence."
)

demo.launch(share=True)